In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, Dense, GlobalAveragePooling3D, Dropout
)
from tensorflow.keras.regularizers import l2

# Load dataset
X_train = np.load('/path/to/training/data/X_train.npy')  # Update path
Y_train = np.load('/path/to/training/data/Y_train.npy')  # Update path

# Expand dimensions to match 3D CNN input: (batch, depth, height, width, channels)
X_train = np.expand_dims(X_train, axis=1)  # Add depth dimension -> (529, 1, 26, 21, 3)

# Define DenseNet-3D block
def dense_block(x, filters, dropout_rate=0.2, l2_lambda=0.01):
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv3D(filters, kernel_size=(3, 3, 3), padding='same', kernel_regularizer=l2(l2_lambda))(x)
    x = Dropout(dropout_rate)(x)
    return x

# Build DenseNet-3D Model
def build_densenet_3d(input_shape):
    inputs = Input(shape=input_shape)

    x = Conv3D(32, kernel_size=(3, 3, 3), padding='same', kernel_regularizer=l2(0.01))(inputs)
    x = dense_block(x, 64)
    x = dense_block(x, 128)
    x = dense_block(x, 256)

    x = GlobalAveragePooling3D()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='linear')(x)  # Regression output

    model = Model(inputs, outputs)
    return model

# Create model
input_shape = (1, 26, 21, 3)  # (depth, height, width, channels)
model = build_densenet_3d(input_shape)

# Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='mse',  # Mean Squared Error for regression
              metrics=['mae'])  # Mean Absolute Error

# Train model (only on training data)
history = model.fit(X_train, Y_train, epochs=50, batch_size=16, verbose=1)

# Print summary
model.summary()


In [ ]:
model.save("trained_densenet3d_model.keras")


In [ ]:
import os

# Save the trained model
model_save_path = "trained_densenet3d_model.keras"
model.save(model_save_path)

# Get absolute path
absolute_path = os.path.abspath(model_save_path)

# Print the directory
print(f"✅ Model saved at: {absolute_path}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load validation dataset
X_val = np.load('/path/to/validation/data/X_val.npy')  # Update with actual path
Y_val = np.load('/path/to/validation/data/Y_val.npy')  # Update with actual path

# Expand dimensions to match model input: (batch, depth, height, width, channels)
X_val = np.expand_dims(X_val, axis=1)  # Adds depth dimension

# Load the trained model
model = tf.keras.models.load_model('/path/to/trained/trained_densenet3d_model.keras')  # Update path

# Predict Y values for validation data
Y_pred = model.predict(X_val)

# Compute squared error for each sample
squared_errors = np.square(Y_pred.flatten() - Y_val)

# Compute Mean Squared Error (MSE) over all validation samples
mse_val = np.mean(squared_errors)

# Print results
print(f"Mean Squared Error on Validation Set: {mse_val:.6f}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load training dataset
X_train = np.load('/path/to/training/data/X_train.npy')  # Update with actual path
Y_train = np.load('/path/to/training/data/Y_train.npy')  # Update with actual path

# Expand dimensions to match model input: (batch, depth, height, width, channels)
X_train = np.expand_dims(X_train, axis=1)  # Adds depth dimension

# Load the trained model
model = tf.keras.models.load_model('/path/to/trained/trained_densenet3d_model.keras')  # Update path

# Predict Y values for training data
Y_pred = model.predict(X_train)

# Compute squared error for each sample
squared_errors = np.square(Y_pred.flatten() - Y_train)

# Compute Mean Squared Error (MSE) over all training samples
mse_train = np.mean(squared_errors)

# Print results
print(f"Mean Squared Error on Training Set: {mse_train:.6f}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load testing dataset
X_test = np.load('/path/to/testing/data/X_test.npy')  # Update with actual path
Y_test = np.load('/path/to/testing/data/Y_test.npy')  # Update with actual path

# Expand dimensions to match model input: (batch, depth, height, width, channels)
X_test = np.expand_dims(X_test, axis=1)  # Adds depth dimension

# Load the trained model
model = tf.keras.models.load_model('/path/to/trained/trained_densenet3d_model.keras')  # Update path

# Predict Y values for testing data
Y_pred = model.predict(X_test)

# Compute RMSE
rmse_test = np.sqrt(np.mean(np.square(Y_pred.flatten() - Y_test)))

# Print results
print(f"Root Mean Squared Error (RMSE) on Test Set: {rmse_test:.6f}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import r2_score

# Load testing dataset
X_test = np.load('/path/to/testing/data/X_test.npy')  # Update with actual path
Y_test = np.load('/path/to/testing/data/Y_test.npy')  # Update with actual path

# Expand dimensions to match model input: (batch, depth, height, width, channels)
X_test = np.expand_dims(X_test, axis=1)  # Adds depth dimension

# Load the trained model
model = tf.keras.models.load_model('/path/to/trained/trained_densenet3d_model.keras')  # Update path

# Predict Y values for testing data
Y_pred = model.predict(X_test)

# Compute R² Score
r2 = r2_score(Y_test, Y_pred)

print(f"Coefficient of Determination (R²) on Testing Set: {r2:.4f}")
